In [1]:
NAMES = ['Angela DiAndreth', 'Sergey Dutt', 'Edward Lombo-Becerra']

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt

In [3]:
from fix_data import get_random_subset, load_and_center_data  

# Project Title: Finding an Unbiased Pendulum Model for Parameter-Estimation Simulations

# Introduction: 

Many common engineering components move according to simple harmonic motion (pendulums, springs, strings, and wires). When designing systems with these parts, you often need their oscillations to decay at a specific rate. To aid in hardware sizing, you can input this desired decay target into a parameter simulation, which directly outputs physical specifications like length and mass.

Here is an example problem: Consider an inertia-driven semi-automatic shotgun used in trap and skeet competitions. In this system, the bolt body acts as a floating mass (m), and an inertia spring of length L sits compressed between the bolt body and the bolt carrier. Firing a standard 12-gauge shell imparts an average force of 1,250N into the spring, pushing the bolt carrier backward to unlock the action, eject the spent shell, and chamber a new round.
To achieve a target fire rate of 4 rounds per second (240RPM) without jamming or misfiring, the spring’s oscillation must decay by $99\%$ before the next shot. Based on this fire rate and safety margin, the spring requires a decay constant of at least 36.84s-1. Additionally, the inertia spring can be no longer than 8cm and the bolt no heavier than 500g. The simulation must determine the optimal spring length ($L$), bolt mass ($m$), and spring stiffness ($k$) to meet these goals.

For this simulation to work, the math model must rely on unbiased physical parameters. This means any uncertainty in a parameter like length ($L$) must come purely from random noise, rather than absorbing the unaccounted physics of another variable. The goal of this project is to use experimental pendulum data to calibrate a simple harmonic model that accurately tracks trajectory data while keeping its physical parameters completely unbiased. If we are successful in finding a model that matches both criteria, we may attempt the example problem from the previous paragraph. 


The differential equation for simple harmonic motion takes the standard form:

$$
\ddot{x} + a_1 \dot{x} + a_2 x = 0
$$

with the analytical solution:

$$
x(t) = e^{-\alpha t} \left( A \cos(\beta t) + B \sin(\beta t) \right)
$$

This following is how the differential equation for a pendulum decaying due to linear air resistance takes the form of simple harmonic motion. Beginning with the Euler-Lagrange equation derived from Hamilton's Principle of Least Action in classical mechanics:

$$
\frac{d}{dt} \left( \frac{\partial \mathcal{L}}{\partial \dot{q}_j} \right) - \frac{\partial \mathcal{L}}{\partial q_j} = 0
$$

The Lagrangian ($\mathcal{L}$) represents the total kinetic energy ($T$) minus the total potential energy ($U$) of the system:
$$
\mathcal{L} = T - U
$$

In Cartesian coordinates, $T = \frac{1}{2} m v^2$ and $U = mgh$. For a pendulum, its height relative to its lowest point is $h = L(1 - \cos\theta)$, where $\theta$ is the angular displacement from equilibrium. Similarly, its linear velocity $v$ is the time derivative of its arc length $s = L \theta$, yielding $v = L \dot{\theta}$.

Plugging these into the Lagrangian yields:

$$
\mathcal{L} = \frac{1}{2} m L^2 \dot{\theta}^2 + mgL(1 - \cos\theta)
$$

However, the conservative Lagrangian formulation only accounts for restorative forces and neglects energy loss. To model the motion of our decaying pendulum, we introduce non-conservative dissipation by replacing the right-hand side of the Euler-Lagrange equation with a linear Rayleigh drag force:

$$
\frac{d}{dt} \left( \frac{\partial \mathcal{L}}{\partial \dot{\theta}} \right) - \frac{\partial \mathcal{L}}{\partial \theta} = -c \dot{\theta}
$$

Evaluating the derivatives gives:

$$
m L^2 \ddot{\theta} + c \dot{\theta} + mgL \sin\theta = 0
$$

Dividing by the rotational inertia term $m L^2$ puts the model into standard form:

$$
\ddot{\theta} + a_1 \dot{\theta} + a_2 \sin\theta = 0
$$

where:

$$
a_1 = \frac{c}{m L^2}, \quad a_2 = \frac{g}{L}
$$

After applying the small-angle Taylor series approximation $\sin\theta \approx \theta$, we arrive directly at the differential equation for simple harmonic motion.

# Methods:

We can determine the optimal physical parameters $a_1$ and $a_2$ with linear regression. Given $m$ data points $(t_m, \theta_m)$, the matrix formulation is given by:

$$
\begin{bmatrix} \ddot{\theta}(t_1) \\ \vdots \\ \ddot{\theta}(t_m) \end{bmatrix} = \begin{bmatrix} -\dot{\theta}(t_1) & -\sin(\theta(t_1)) \\ \vdots & \vdots \\ -\dot{\theta}(t_m) & -\sin(\theta(t_m)) \end{bmatrix} \begin{bmatrix} a_1 \\ a_2 \end{bmatrix}
$$

Because we only have discrete measurements of $\theta$, we need a continuous, smooth, and differentiable function of the trajectory. Once established, automatic differentiation (JAX) is applied directly to obtain derivatives $\dot{\theta}(t)$ and $\ddot{\theta}(t)$.

Assuming that $\theta_{\text{data}}(t) = \theta(t) + \eta$, where $\eta$ represents a random error, the non-linear model of motion becomes:

$$
\ddot{\theta} + a_1 \dot{\theta} + a_2 \sin(\theta + \eta) = 0
$$

Expanding $\sin(\theta + \eta)$ with a Taylor series yields:

$$
\sin(\theta + \eta) = \sin(\eta) + \cos(\eta)\theta - \frac{\sin(\eta)}{2!}\theta^2 - \frac{\cos(\eta)}{3!}\theta^3 + \frac{\sin(\eta)}{4!}\theta^4 + \dots
$$

The general solution for the linearly damped harmonic oscillator subject to initial conditions $\theta(0) = \theta_{\text{max}}$ and $\dot{\theta}(0) = 0$ is:

$$
\theta(t) = e^{-\alpha t} \left( \theta_{\text{max}} \cos(\beta t) + \frac{\alpha}{\beta} \theta_{\text{max}} \sin(\beta t) \right)
$$

Given the long length and slow decay rate of this pendulum, the natural frequency $\beta \sim \mathcal{O}(10^0)\text{ rad/s}$ is significantly larger than the damping coefficient $\alpha \sim \mathcal{O}(10^{-2})\text{ s}^{-1}$. Thus, the relative ratio evaluates to:

$$
\frac{\alpha}{\beta} \sim \mathcal{O}\left(10^{-2}\right) \ll 1
$$

Over any single oscillation period, the contribution from the sine term is negligible, allowing the trajectory to be approximated as:

$$
\theta(t) \approx \theta_{\text{max}} e^{-\alpha t} \cos(\beta t)
$$

Substituting this approximation into the Taylor series expansion and applying  trigonometric power-reduction identities (e.g., $\cos^2(x) = \frac{1}{2} + \frac{1}{2}\cos(2x)$, $\cos^3(x) = \frac{3}{4}\cos(x) + \frac{1}{4}\cos(3x)$) allows the non-linear force to be expressed strictly as a series:
$$
\sin(\theta + \eta) \approx e^{-\alpha t} \sum_{k=0}^{\infty} A_k \cos(k \beta t)
$$

Additionally, ($A_0 \approx 0$) while $A_1 \approx \theta_{\text{max}}$. Rearranging terms allows the full non-linear differential equation to be rewritten as:

$$
\ddot{\theta} + a_1 \dot{\theta} + a_2 \theta = f(t)
$$

$$
f(t) = -a_2 e^{-\alpha t} \sum_{k=2}^{\infty} A_k \cos(k \beta t)
$$

The general solution to a linear non-homogeneous differential equation is the sum of the homogeneous solution $\theta_h(t)$ and the particular solution $\theta_p(t)$:

$$
\theta(t) = \theta_h(t) + \theta_p(t)
$$

We already know $\theta_h(t)$, and it turns out that:

$$
\theta_p(t) = e^{-\alpha t} \sum_{k=2}^{\infty} \left[ A_k \cos(k \beta t) + B_k \sin(k \beta t) \right]
$$

Combining these yields a continuous, differentiable function expressed as a damped series:

$$
\theta(t) = e^{-\alpha t} \sum_{k=1}^{\infty} \left[ A_k \cos(k \beta t) + B_k \sin(k \beta t) \right]
$$

For $m$ data points truncated at $N$ iterations, the coefficients $(A_k, B_k)$ are fitted using the linear system:

$$\begin{bmatrix} \theta(t_1) \\ \vdots \\ \theta(t_m) \end{bmatrix} = \begin{bmatrix} e^{-\alpha t_1}\cos(1\beta t_1) & e^{-\alpha t_1}\sin(1\beta t_1) & \dots & e^{-\alpha t_1}\cos(N\beta t_1) & e^{-\alpha t_1}\sin(N\beta t_1) \\ \vdots & \vdots & \ddots & \vdots & \vdots \\ e^{-\alpha t_m}\cos(1\beta t_m) & e^{-\alpha t_m}\sin(1\beta t_m) & \dots & e^{-\alpha t_m}\cos(N\beta t_m) & e^{-\alpha t_m}\sin(N\beta t_m) \end{bmatrix} \begin{bmatrix} A_1 \\ B_1 \\ \vdots \\ A_N \\ B_N \end{bmatrix}
$$

To prevent overfitting, Model accuracy is evaluated using the Mean Squared Error (MSE):
$$
\text{MSE} = \frac{1}{m} \sum_{i=1}^{m} \left( \theta_{\text{data}}(t_i) - \theta(t_i) \right)^2
$$

We stop adding terms to the series when the relative improvement in MSE drops below a pre-set tolerance threshold:

$$
\frac{\text{MSE}(N-1) - \text{MSE}(N)}{\text{MSE}(N-1)} < \text{tol}, \quad \text{where } \text{tol} = 0.01\text{ (1\%)}
$$

# Project Code: